# NewsBot 2.0 | Multilingual Semantic Search Laboratory

**ITAI 2373 final project, Phase 1**

This Google Colab notebook executes genuine shared vector semantic retrieval using a public multilingual Sentence Transformers model. The six article paragraphs and 18 evaluation queries are **synthetic teaching examples**, not BBC news or a live news API. The corpus and source texts are openly displayed for reproducibility.

## Learning outcomes
1. Detect English, Spanish and French.
2. Encode English articles and queries in three languages in a common vector space.
3. Compare TF IDF baseline and multilingual MiniLM Recall@1 and Recall@3.
4. Optionally use a third party translator with informed consent.
5. Identify evidence limitations and translation failures.

**Compute notice:** model download uses the internet and may require more resources than Render Free provides. Run this notebook in Google Colab or a suitable local computer. No real BBC article text is embedded here.

In [ ]:
%pip -q install "sentence-transformers>=3,<6" "langdetect>=1,<2" "deep-translator>=1.11,<2" "scikit-learn>=1.4,<2" pandas numpy matplotlib

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0
MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_ID)
print("Real multilingual encoder loaded:", MODEL_ID)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

## Synthetic benchmark corpus
Six English passages about technology, climate, finance, health, sports and cinema, with English, Spanish and French questions prepared independently. These examples are only an educational sanity test: they are too small for population level accuracy claims.

In [ ]:
documents = [
    ("robotics", "Engineers designed autonomous robotics systems and machine learning algorithms for a factory."),
    ("climate", "Researchers documented climate change and renewable solar energy for coastal cities."),
    ("finance", "A financial institution reported quarterly profits, revenue growth and share prices."),
    ("health", "Doctors studied hospital treatments and vaccines to improve patient health."),
    ("sport", "A football club won the tournament after its final match against a rival team."),
    ("cinema", "An award winning movie director announced a new international film release."),
]
queries = [
    ("en", "robotics", "How are robots and machine learning used in a factory?"),
    ("es", "robotics", "¿Cómo se utilizan los robots y el aprendizaje automático en una fábrica?"),
    ("fr", "robotics", "Comment les robots et l’apprentissage automatique sont ils utilisés dans une usine ?"),
    ("en", "climate", "What do scientists say about renewable solar energy?"),
    ("es", "climate", "¿Qué dicen los científicos sobre la energía solar renovable?"),
    ("fr", "climate", "Que disent les scientifiques au sujet de l’énergie solaire renouvelable ?"),
    ("en", "finance", "What did the bank report about profits and revenues?"),
    ("es", "finance", "¿Qué informó el banco sobre sus beneficios e ingresos?"),
    ("fr", "finance", "Que rapporte la banque au sujet des bénéfices et des revenus ?"),
    ("en", "health", "Which hospital treatment improved patient health?"),
    ("es", "health", "¿Qué tratamientos hospitalarios mejoraron la salud de los pacientes?"),
    ("fr", "health", "Quels traitements hospitaliers améliorent la santé des patients ?"),
    ("en", "sport", "Which team won the football tournament final?"),
    ("es", "sport", "¿Qué equipo ganó la final del torneo de fútbol?"),
    ("fr", "sport", "Quelle équipe a remporté la finale du tournoi de football ?"),
    ("en", "cinema", "Which director announced the international film?"),
    ("es", "cinema", "¿Qué director anunció una película internacional?"),
    ("fr", "cinema", "Quel réalisateur a annoncé un nouveau film international ?"),
]
df = pd.DataFrame(queries, columns=["language", "expected_article", "query"])
df["language_detected"] = df["query"].map(detect)
print("Language detection agreement:", (df.language == df.language_detected).mean())
display(df.head(6))

## True multilingual embedding retrieval versus lexical search
Both the query and English article passage are encoded by the same pretrained multilingual neural model. A rank is considered correct only when the synthetic target passage appears at the selected position. TF IDF is a lexical baseline and does not translate queries.

In [ ]:
ids = [x[0] for x in documents]
texts = [x[1] for x in documents]
q_texts = df["query"].tolist()
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
text_tfidf = vectorizer.fit_transform(texts)
lexical = cosine_similarity(vectorizer.transform(q_texts), text_tfidf)
emb_documents = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
emb_queries = model.encode(q_texts, normalize_embeddings=True, show_progress_bar=False)
semantic = cosine_similarity(emb_queries, emb_documents)
records = []
for i, q in df.iterrows():
    for label, matrix in [("TF IDF baseline", lexical), ("Multilingual MiniLM", semantic)]:
        ranking = np.argsort(matrix[i])[::-1]
        rank = next((j + 1 for j, idx in enumerate(ranking) if ids[idx] == q.expected_article), None)
        records.append({"language": q.language, "method": label, "query": q.query,
                        "expected": q.expected_article, "rank": rank,
                        "recall_at_1": int(rank is not None and rank <= 1),
                        "recall_at_3": int(rank is not None and rank <= 3)})
results = pd.DataFrame(records)
summary = results.groupby(["method", "language"], as_index=False)[["recall_at_1", "recall_at_3"]].mean()
display(summary)
ax = summary.pivot(index="language",columns="method",values="recall_at_1").plot(kind="bar",figsize=(8,4),title="Measured Recall@1 on 18 synthetic teaching queries")
ax.set_ylim(0,1.05)
ax.set_ylabel("Recall at 1")
plt.tight_layout()
plt.show()

In [ ]:
# Display examples of genuine cross-language ranking from the trained encoder.
for language in ("en", "es", "fr"):
    q = df[df.language==language].iloc[0]
    idx = df.index[df.language==language][0]
    order = np.argsort(semantic[idx])[::-1]
    print(language, "query:", q.query)
    print("Top result:", ids[int(order[0])], "similarity:", round(float(semantic[idx,order[0]]),3))
    print()

## Translation with informed consent
The default below is **disabled**. Setting the consent variable to True sends the selected non confidential sentence to a third party translation provider. This is separate from local neural retrieval. Translation quality and service availability must not be assumed.

In [ ]:
ALLOW_EXTERNAL_TRANSLATION = False  # Only change after reviewing service privacy implications.
if ALLOW_EXTERNAL_TRANSLATION:
    from deep_translator import GoogleTranslator
    example = "El gobierno anunció un programa para nuevas tecnologías."
    try:
        translated = GoogleTranslator(source="es", target="en").translate(example)
        print("Translation:", translated)
        print("Verify named entities, negation and nuances manually before reporting quality.")
    except Exception as exc:
        print("Translation provider unavailable:", type(exc).__name__)
else:
    print("External translation disabled. Dense multilingual retrieval did not require translation.")

## Research notes and limitations
* The six English passages and 18 queries are constructed examples; evaluation numbers are measured by this run but **not** generalizable to multilingual BBC news.
* Cross-language retrieval is neural semantic search; translation-assisted lexical/LSA search is a separate method.
* Language detection on short/code switched text can fail.
* Dense similarity does not verify the article facts or culturally specific perspective.
* The full ITAI 2373 rubric also requests comparative multilingual coverage and cultural context; neither can be validated using the uploaded English BBC article corpus alone.

### Suggested evidence for final technical documentation
Save the model identifier, package versions, per-language Recall@1 and Recall@3, the full ranking table, errors, and one screenshot. Mark the source as a *small synthetic validation set*.

In [ ]:
import json, platform
import sentence_transformers
from pathlib import Path
out=Path('newsbot_multilingual_evaluation')
out.mkdir(exist_ok=True)
results.to_csv(out/'retrieval_rankings.csv',index=False)
summary.to_csv(out/'summary.csv',index=False)
meta={"model": MODEL_ID, "python":platform.python_version(), "sentence_transformers":sentence_transformers.__version__,
      "documents":len(documents), "queries":len(queries), "dataset":"synthetic controlled examples",
      "no_real_bbc_performance_claim":True}
(out/'manifest.json').write_text(json.dumps(meta,indent=2))
print('Evidence saved in',out.resolve())